# 🔌 Notebook 2: Circuit Breaker

Retries are great for *transient* failures. But if the downstream is **clearly dead** (fire in the datacenter, crash loop, bad deploy), retrying just wastes CPU, exhausts connection pools, and slows down the caller's own recovery.

A **circuit breaker** watches recent failures. After too many, it *trips* and rejects calls immediately — without touching the downstream — until a cooldown elapses and it cautiously probes again.

**States:**
- 🟢 **CLOSED** — normal, calls pass through.
- 🔴 **OPEN** — downstream is sick, fail fast without calling it.
- 🟡 **HALF_OPEN** — cooldown elapsed, send one probe to test the waters.

Named after the breaker in your electrical panel: it trips when there's a fault, saves the house, and you reset it once the fault is fixed.

## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🎯 The slow, dead downstream

We simulate a downstream where each call takes 200ms before failing (like a TCP connect timeout). Without protection, every caller pays that cost.

In [ ]:
import time

broken = True

def downstream():
    time.sleep(0.2)       # simulates a slow failure (timeout)
    if broken:
        raise IOError('500 service unavailable')
    return 'ok'


## 🟥 BAD: no breaker — keep hammering

Every single call pays the 200 ms timeout, *even though we already know the service is dead*. With 1,000 clients, that's 1,000 × 200 ms of wasted threads and connections per round.

In [ ]:
t0 = time.perf_counter()
errors = 0
for i in range(10):
    try:
        downstream()
    except IOError:
        errors += 1
print(f'{errors} failures in {time.perf_counter()-t0:.2f}s — every call blocked on the slow failure')


## 🟩 GOOD: circuit breaker

After `fail_threshold` consecutive failures, the breaker **opens** and future calls fail *instantly*. After `cooldown`, it goes **half-open** and lets one probe through:
- probe succeeds → back to CLOSED (service recovered).
- probe fails → back to OPEN (wait another cooldown).

In [ ]:
class CircuitBreaker:
    def __init__(self, fail_threshold=3, cooldown=1.0):
        self.fail_threshold = fail_threshold
        self.cooldown = cooldown
        self.fails = 0
        self.opened_at = 0.0
        self.state = 'CLOSED'

    def call(self, fn):
        # 1. If OPEN, check if the cooldown elapsed.
        if self.state == 'OPEN':
            if time.monotonic() - self.opened_at >= self.cooldown:
                self.state = 'HALF_OPEN'
                print('  ↪ probing (HALF_OPEN)')
            else:
                raise RuntimeError('circuit OPEN — fast-failing')

        # 2. Try the call.
        try:
            result = fn()
        except Exception:
            self.fails += 1
            # HALF_OPEN probe fail, or hit threshold: OPEN the circuit.
            if self.state == 'HALF_OPEN' or self.fails >= self.fail_threshold:
                self.state = 'OPEN'
                self.opened_at = time.monotonic()
                print('  ⚡ circuit OPENED')
            raise

        # 3. Success -> reset.
        self.state = 'CLOSED'
        self.fails = 0
        return result


## 💥 Watch it trip, cool down, and recover

In [ ]:
broken = True
cb = CircuitBreaker(fail_threshold=3, cooldown=1.0)

t0 = time.perf_counter()
for i in range(8):
    try:
        print(i, cb.call(downstream))
    except Exception as e:
        print(i, f'[{cb.state}] FAIL:', e)
    time.sleep(0.1)
print(f'elapsed so far: {time.perf_counter()-t0:.2f}s (calls 4-7 fast-failed)')

print('\n--- downstream recovers ---')
broken = False
time.sleep(1.1)  # wait out the cooldown
for i in range(3):
    try:
        print('recovery', i, cb.call(downstream))
    except Exception as e:
        print('recovery', i, 'FAIL:', e)


## 📊 Visualize the latency savings

The table above is nicer as a picture. Let's run **30 back-to-back calls** against the dead downstream and plot the latency of each one, twice:

- **No breaker** — every call pays the full 200 ms timeout.
- **With breaker** — the first few pay the timeout, then the breaker opens and the rest *fast-fail* in microseconds.

We give the breaker a very long cooldown so it stays OPEN for the whole run — that makes the chart deterministic (no surprise probes mid-experiment).


In [ ]:
import matplotlib.pyplot as plt

def measure(call_fn, n=30):
    """Run call_fn() n times; return list of per-call latencies in seconds."""
    latencies = []
    for _ in range(n):
        t = time.perf_counter()
        try: call_fn()
        except Exception: pass
        latencies.append(time.perf_counter() - t)
    return latencies

broken = True

# No-breaker baseline: every call takes ~200ms.
no_breaker = measure(downstream)

# With-breaker: long cooldown so it never half-opens during the run.
cb2 = CircuitBreaker(fail_threshold=3, cooldown=999)
with_breaker = measure(lambda: cb2.call(downstream))

fig, ax = plt.subplots(figsize=(10, 3.2))
x = range(1, len(no_breaker) + 1)
ax.bar([i - 0.2 for i in x], no_breaker,   width=0.4, color='#d97706', label='no breaker')
ax.bar([i + 0.2 for i in x], with_breaker, width=0.4, color='#16a34a', label='with breaker')
ax.set_xlabel('call #'); ax.set_ylabel('latency (s)')
ax.set_title('Per-call latency: breaker collapses to ~0 s after it opens')
ax.legend()
plt.tight_layout(); plt.show()

total_no  = sum(no_breaker)
total_yes = sum(with_breaker)
print(f'total wall-time — no breaker: {total_no:.2f}s   with breaker: {total_yes:.2f}s   ({total_no/total_yes:.0f}× faster)')


Notice how calls 4-7 returned *instantly* instead of each costing 200 ms — the breaker saved us from the slow failure cascade.

## 🌍 Real-world refinements

Our toy breaker trips on **consecutive failures**. Production breakers (e.g. Netflix's [Hystrix](https://github.com/Netflix/Hystrix), [`pybreaker`](https://github.com/danielfm/pybreaker), [resilience4j](https://resilience4j.readme.io/)) usually trip on a **failure *rate* over a sliding window** — e.g. 'open if >50% of the last 20 calls failed, and we saw at least 10 calls'. That's more robust against noisy intermittent errors.

Other practical tweaks:
- **Exponential cooldown** — each successive trip waits longer before probing.
- **Separate breaker per downstream** — don't let Service B's outage open the breaker for Service C.
- **Combine with fallback** — instead of raising when OPEN, return a cached value or default (we cover this in Notebook 4).
- **Metrics/alerts** — the moment a breaker opens is exactly when you want to page the on-call.
- **Combine with retry** — retry handles transient blips; the breaker catches sustained outages. Use retry *inside* the breaker, not around it.